# 23. SelF Personalized Model Pickle 생성

## 목적
검증된 SVD 임베딩을 프로덕션용 Pickle 파일로 패키징합니다.

## 입력 파일
- `data/processed/self/svd_results.pkl` (21번 노트북 출력)
- `data/processed/self/svd_evaluation_results.pkl` (22번 노트북 출력)

## 출력 파일
- `pred/models/self_personalized_v1.pkl`

## Pickle 구조
```python
{
    'version': '1.0.0',
    'created_at': datetime,
    'metadata': {
        'n_users': int,
        'n_products': int,
        'embedding_dim': int,
        'training_date': str,
        'evaluation_metrics': dict
    },
    'components': {
        'user_embeddings': np.ndarray,
        'product_embeddings': np.ndarray,
        'user_id_to_idx': dict,
        'idx_to_user_id': dict,
        'product_id_to_idx': dict,
        'idx_to_product_id': dict,
        'global_popular_products': list,
        'category_popular_products': dict
    },
    'hyperparameters': {
        'n_components': int,
        'weight_view': float,
        'weight_cart': float,
        'weight_order': float
    }
}
```

In [1]:
# 필수 라이브러리 임포트
import pickle
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime
import json
import warnings
warnings.filterwarnings('ignore')

print("라이브러리 로드 완료")
print(f"현재 시간: {datetime.now()}")

라이브러리 로드 완료
현재 시간: 2025-12-15 02:00:50.310102


## 1. 입력 파일 로드

In [2]:
# 경로 설정
INPUT_DIR = Path('../data/processed/self')
OUTPUT_DIR = Path('../pred/models')

SVD_RESULTS_PATH = INPUT_DIR / 'svd_results.pkl'
EVAL_RESULTS_PATH = INPUT_DIR / 'svd_evaluation_results.pkl'

# 출력 디렉토리 생성
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"입력 디렉토리: {INPUT_DIR}")
print(f"출력 디렉토리: {OUTPUT_DIR}")

입력 디렉토리: ..\data\processed\self
출력 디렉토리: ..\pred\models


In [3]:
# SVD 결과 로드
if not SVD_RESULTS_PATH.exists():
    raise FileNotFoundError(f"SVD 결과 파일이 없습니다: {SVD_RESULTS_PATH}")

with open(SVD_RESULTS_PATH, 'rb') as f:
    svd_results = pickle.load(f)

# 레거시 포맷 호환 처리 (21번 노트북 출력이 평탄한 구조인 경우)
if isinstance(svd_results, dict) and 'embeddings' not in svd_results:
    legacy_required = {
        'user_embeddings', 'product_embeddings',
        'user_id_to_idx', 'idx_to_user_id',
        'product_id_to_idx', 'idx_to_product_id',
        'metadata',
    }

    if legacy_required.issubset(set(svd_results.keys())):
        legacy_metadata = svd_results.get('metadata', {})
        legacy_weights = legacy_metadata.get('weights', {})
        legacy_hyperparameters = {
            'n_components': legacy_metadata.get('n_components', 128),
            'weight_view': float(legacy_weights.get('view', 1.0)),
            'weight_cart': float(legacy_weights.get('cart', 3.0)),
            'weight_order': float(legacy_weights.get('order', 5.0)),
            'log_transform': True,
        }

        svd_results = {
            'embeddings': {
                'user_embeddings': svd_results['user_embeddings'],
                'product_embeddings': svd_results['product_embeddings'],
            },
            'mappings': {
                'user_id_to_idx': svd_results['user_id_to_idx'],
                'idx_to_user_id': svd_results['idx_to_user_id'],
                'product_id_to_idx': svd_results['product_id_to_idx'],
                'idx_to_product_id': svd_results['idx_to_product_id'],
            },
            'metadata': legacy_metadata,
            'popular_products': {
                'global': svd_results.get('global_popular', []),
                'by_category': svd_results.get('category_popular', {}),
            },
            'version': svd_results.get('version', 'legacy'),
            'created_at': svd_results.get('created_at', 'unknown'),
            'hyperparameters': legacy_hyperparameters,
        }
        print("레거시 SVD 결과 포맷을 최신 포맷으로 변환했습니다.")

print(f"SVD 결과 로드 완료: {SVD_RESULTS_PATH}")
print(f"  버전: {svd_results.get('version', 'unknown')}")
print(f"  생성일: {svd_results.get('created_at', 'unknown')}")

레거시 SVD 결과 포맷을 최신 포맷으로 변환했습니다.
SVD 결과 로드 완료: ..\data\processed\self\svd_results.pkl
  버전: legacy
  생성일: unknown


In [4]:
# 평가 결과 로드 (선택적)
if EVAL_RESULTS_PATH.exists():
    with open(EVAL_RESULTS_PATH, 'rb') as f:
        eval_results = pickle.load(f)
    print(f"평가 결과 로드 완료: {EVAL_RESULTS_PATH}")
    EVAL_LOADED = True
else:
    print(f"[경고] 평가 결과 파일이 없습니다: {EVAL_RESULTS_PATH}")
    print("22번 노트북을 먼저 실행하는 것을 권장합니다.")
    eval_results = None
    EVAL_LOADED = False

평가 결과 로드 완료: ..\data\processed\self\svd_evaluation_results.pkl


## 2. 데이터 검증

In [5]:
# 필수 컴포넌트 검증
REQUIRED_KEYS = ['embeddings', 'mappings', 'metadata']
REQUIRED_EMBEDDINGS = ['user_embeddings', 'product_embeddings']
REQUIRED_MAPPINGS = ['user_id_to_idx', 'idx_to_user_id', 'product_id_to_idx', 'idx_to_product_id']

print("필수 컴포넌트 검증:")

# 최상위 키 검증
for key in REQUIRED_KEYS:
    if key in svd_results:
        print(f"  ✅ {key}")
    else:
        print(f"  ❌ {key} - 누락됨")
        raise KeyError(f"필수 키 누락: {key}")

# 임베딩 검증
for key in REQUIRED_EMBEDDINGS:
    if key in svd_results['embeddings']:
        shape = svd_results['embeddings'][key].shape
        print(f"  ✅ embeddings.{key} {shape}")
    else:
        print(f"  ❌ embeddings.{key} - 누락됨")
        raise KeyError(f"필수 임베딩 누락: {key}")

# 매핑 검증
for key in REQUIRED_MAPPINGS:
    if key in svd_results['mappings']:
        length = len(svd_results['mappings'][key])
        print(f"  ✅ mappings.{key} ({length} entries)")
    else:
        print(f"  ❌ mappings.{key} - 누락됨")
        raise KeyError(f"필수 매핑 누락: {key}")

print("\n모든 필수 컴포넌트 검증 완료")

필수 컴포넌트 검증:
  ✅ embeddings
  ✅ mappings
  ✅ metadata
  ✅ embeddings.user_embeddings (5000, 128)
  ✅ embeddings.product_embeddings (1995, 128)
  ✅ mappings.user_id_to_idx (5000 entries)
  ✅ mappings.idx_to_user_id (5000 entries)
  ✅ mappings.product_id_to_idx (1995 entries)
  ✅ mappings.idx_to_product_id (1995 entries)

모든 필수 컴포넌트 검증 완료


In [6]:
# 데이터 일관성 검증
user_embeddings = svd_results['embeddings']['user_embeddings']
product_embeddings = svd_results['embeddings']['product_embeddings']
user_id_to_idx = svd_results['mappings']['user_id_to_idx']
product_id_to_idx = svd_results['mappings']['product_id_to_idx']

print("데이터 일관성 검증:")

# 사용자 수 일치 확인
n_users_embed = user_embeddings.shape[0]
n_users_map = len(user_id_to_idx)
if n_users_embed == n_users_map:
    print(f"  ✅ 사용자 수 일치: {n_users_embed}")
else:
    print(f"  ❌ 사용자 수 불일치: 임베딩={n_users_embed}, 매핑={n_users_map}")
    raise ValueError("사용자 수 불일치")

# 제품 수 일치 확인
n_products_embed = product_embeddings.shape[0]
n_products_map = len(product_id_to_idx)
if n_products_embed == n_products_map:
    print(f"  ✅ 제품 수 일치: {n_products_embed}")
else:
    print(f"  ❌ 제품 수 불일치: 임베딩={n_products_embed}, 매핑={n_products_map}")
    raise ValueError("제품 수 불일치")

# 임베딩 차원 일치 확인
user_dim = user_embeddings.shape[1]
product_dim = product_embeddings.shape[1]
if user_dim == product_dim:
    print(f"  ✅ 임베딩 차원 일치: {user_dim}")
else:
    print(f"  ❌ 임베딩 차원 불일치: 사용자={user_dim}, 제품={product_dim}")
    raise ValueError("임베딩 차원 불일치")

# NaN/Inf 확인
if np.isnan(user_embeddings).any() or np.isinf(user_embeddings).any():
    print("  ❌ 사용자 임베딩에 NaN/Inf 존재")
    raise ValueError("사용자 임베딩에 NaN/Inf 존재")
else:
    print("  ✅ 사용자 임베딩 NaN/Inf 없음")

if np.isnan(product_embeddings).any() or np.isinf(product_embeddings).any():
    print("  ❌ 제품 임베딩에 NaN/Inf 존재")
    raise ValueError("제품 임베딩에 NaN/Inf 존재")
else:
    print("  ✅ 제품 임베딩 NaN/Inf 없음")

print("\n데이터 일관성 검증 완료")

데이터 일관성 검증:
  ✅ 사용자 수 일치: 5000
  ✅ 제품 수 일치: 1995
  ✅ 임베딩 차원 일치: 128
  ✅ 사용자 임베딩 NaN/Inf 없음
  ✅ 제품 임베딩 NaN/Inf 없음

데이터 일관성 검증 완료


## 3. 인기 제품 목록 생성

In [7]:
# 글로벌 인기 제품 (이미 생성된 경우 사용)
if 'popular_products' in svd_results:
    global_popular_products = svd_results['popular_products'].get('global', [])
    category_popular_products = svd_results['popular_products'].get('by_category', {})
    print(f"기존 인기 제품 목록 사용")
    print(f"  글로벌 인기 제품: {len(global_popular_products)}개")
    print(f"  카테고리별 인기 제품: {len(category_popular_products)}개 카테고리")
else:
    # 인기 제품 목록이 없으면 기본값 생성
    print("인기 제품 목록 생성 중...")
    
    # 모든 제품을 글로벌 인기 목록으로 (순서대로)
    idx_to_product_id = svd_results['mappings']['idx_to_product_id']
    global_popular_products = [idx_to_product_id[i] for i in range(len(idx_to_product_id))]
    
    # 카테고리별은 빈 딕셔너리 (실제 데이터 필요)
    category_popular_products = {}
    
    print(f"  글로벌 인기 제품: {len(global_popular_products)}개 (전체 제품)")
    print(f"  카테고리별 인기 제품: 미생성 (실제 DB 데이터 필요)")

기존 인기 제품 목록 사용
  글로벌 인기 제품: 100개
  카테고리별 인기 제품: 10개 카테고리


## 4. Pickle 구조 생성

In [8]:
# Pickle 구조 생성
MODEL_VERSION = '1.0.0'

# 하이퍼파라미터 (21번 노트북과 동일)
hyperparameters = svd_results.get('hyperparameters', {
    'n_components': 128,
    'weight_view': 1.0,
    'weight_cart': 3.0,
    'weight_order': 5.0,
    'log_transform': True
})

# 평가 메트릭
if EVAL_LOADED:
    evaluation_metrics = {
        'same_category_ratio': eval_results['product_similarity']['same_category_ratio'],
        'hit_at_10': eval_results['holdout_metrics']['hit_at_10'],
        'ndcg_at_10': eval_results['holdout_metrics']['ndcg_at_10'],
        'all_passed': all([
            eval_results['product_similarity']['passed'],
            eval_results['holdout_metrics']['hit_passed'],
            eval_results['holdout_metrics']['ndcg_passed']
        ])
    }
else:
    evaluation_metrics = {
        'note': '평가 미실행 - 22번 노트북 실행 권장'
    }

# numpy 버전 호환성 처리
# numpy 2.x에서 저장된 배열은 numpy 1.x에서 로드 불가 (numpy._core 모듈 문제)
# 해결책: numpy 배열을 표준 Python bytes로 저장하고 로드 시 복원
def save_numpy_compatible(arr):
    """numpy 배열을 버전 호환 가능한 형태로 변환"""
    return {
        'data': arr.tobytes(),
        'shape': arr.shape,
        'dtype': str(arr.dtype)
    }

# 최종 Pickle 구조
self_personalized_model = {
    'version': MODEL_VERSION,
    'created_at': datetime.now(),
    'model_type': 'self_personalized_svd',
    'description': 'SelF 개인화 추천 모델 (Truncated SVD 기반 협업 필터링)',
    
    'metadata': {
        'n_users': user_embeddings.shape[0],
        'n_products': product_embeddings.shape[0],
        'embedding_dim': user_embeddings.shape[1],
        'training_date': svd_results.get('created_at', datetime.now()).strftime('%Y-%m-%d') if hasattr(svd_results.get('created_at', datetime.now()), 'strftime') else str(svd_results.get('created_at', 'unknown')),
        'source_data': 'SelF user_product_stats',
        'evaluation_metrics': evaluation_metrics,
        'numpy_compatible': True  # 호환성 플래그
    },
    
    'components': {
        # 임베딩 (numpy 버전 호환 형식으로 저장)
        'user_embeddings': save_numpy_compatible(user_embeddings),
        'product_embeddings': save_numpy_compatible(product_embeddings),
        
        # ID ↔ 인덱스 매핑 (일반 dict는 호환성 문제 없음)
        'user_id_to_idx': {int(k): int(v) for k, v in svd_results['mappings']['user_id_to_idx'].items()},
        'idx_to_user_id': {int(k): int(v) for k, v in svd_results['mappings']['idx_to_user_id'].items()},
        'product_id_to_idx': {int(k): int(v) for k, v in svd_results['mappings']['product_id_to_idx'].items()},
        'idx_to_product_id': {int(k): int(v) for k, v in svd_results['mappings']['idx_to_product_id'].items()},
        
        # 인기 제품 (폴백용) - int로 변환하여 numpy 의존성 제거
        'global_popular_products': [int(x) for x in global_popular_products],
        'category_popular_products': {int(k): [int(x) for x in v] for k, v in category_popular_products.items()}
    },
    
    'hyperparameters': hyperparameters
}

print("Pickle 구조 생성 완료 (numpy 버전 호환 형식)")
print(f"\n모델 정보:")
print(f"  버전: {MODEL_VERSION}")
print(f"  사용자 수: {self_personalized_model['metadata']['n_users']}")
print(f"  제품 수: {self_personalized_model['metadata']['n_products']}")
print(f"  임베딩 차원: {self_personalized_model['metadata']['embedding_dim']}")
print(f"  numpy 호환 형식: True")

Pickle 구조 생성 완료 (numpy 버전 호환 형식)

모델 정보:
  버전: 1.0.0
  사용자 수: 5000
  제품 수: 1995
  임베딩 차원: 128
  numpy 호환 형식: True


## 5. Pickle 저장

In [9]:
# Pickle 파일 저장
# 주의: numpy 1.x/2.x 호환성을 위해 protocol=4 사용 (Python 3.8+)
# pickle.HIGHEST_PROTOCOL은 numpy 2.x에서 생성시 numpy 1.x에서 로드 불가
OUTPUT_PATH = OUTPUT_DIR / 'self_personalized_v1.pkl'

# numpy 배열을 리스트로 변환하여 호환성 확보 (선택적)
# 또는 protocol=4 사용으로 대부분의 경우 호환성 확보
with open(OUTPUT_PATH, 'wb') as f:
    pickle.dump(self_personalized_model, f, protocol=4)

# 파일 크기 확인
file_size_mb = OUTPUT_PATH.stat().st_size / (1024 * 1024)

print(f"Pickle 저장 완료: {OUTPUT_PATH}")
print(f"파일 크기: {file_size_mb:.2f} MB")
print(f"Pickle protocol: 4 (Python 3.8+ 호환)")

Pickle 저장 완료: ..\pred\models\self_personalized_v1.pkl
파일 크기: 3.50 MB
Pickle protocol: 4 (Python 3.8+ 호환)


## 6. 저장된 Pickle 검증

In [10]:
# 저장된 파일 다시 로드하여 검증
print("저장된 Pickle 파일 검증 중...")

with open(OUTPUT_PATH, 'rb') as f:
    loaded_model = pickle.load(f)

# numpy 배열 복원 함수
def load_numpy_compatible(data_dict):
    """numpy 버전 호환 형식에서 배열 복원"""
    if isinstance(data_dict, dict) and 'data' in data_dict and 'shape' in data_dict:
        return np.frombuffer(data_dict['data'], dtype=data_dict['dtype']).reshape(data_dict['shape'])
    return data_dict  # 이미 numpy 배열인 경우 (구버전 pickle)

# 구조 검증
REQUIRED_TOP_KEYS = ['version', 'created_at', 'metadata', 'components', 'hyperparameters']
REQUIRED_COMPONENT_KEYS = [
    'user_embeddings', 'product_embeddings',
    'user_id_to_idx', 'idx_to_user_id',
    'product_id_to_idx', 'idx_to_product_id',
    'global_popular_products'
]

print("\n구조 검증:")
for key in REQUIRED_TOP_KEYS:
    if key in loaded_model:
        print(f"  [OK] {key}")
    else:
        print(f"  [FAIL] {key}")

print("\n컴포넌트 검증:")
for key in REQUIRED_COMPONENT_KEYS:
    if key in loaded_model['components']:
        print(f"  [OK] components.{key}")
    else:
        print(f"  [FAIL] components.{key}")

# numpy 배열 복원 테스트
user_emb_restored = load_numpy_compatible(loaded_model['components']['user_embeddings'])
prod_emb_restored = load_numpy_compatible(loaded_model['components']['product_embeddings'])

print(f"\n임베딩 복원 테스트:")
print(f"  user_embeddings shape: {user_emb_restored.shape}")
print(f"  product_embeddings shape: {prod_emb_restored.shape}")

저장된 Pickle 파일 검증 중...

구조 검증:
  [OK] version
  [OK] created_at
  [OK] metadata
  [OK] components
  [OK] hyperparameters

컴포넌트 검증:
  [OK] components.user_embeddings
  [OK] components.product_embeddings
  [OK] components.user_id_to_idx
  [OK] components.idx_to_user_id
  [OK] components.product_id_to_idx
  [OK] components.idx_to_product_id
  [OK] components.global_popular_products

임베딩 복원 테스트:
  user_embeddings shape: (5000, 128)
  product_embeddings shape: (1995, 128)


In [11]:
# numpy 배열 복원 함수 (추천에서 사용)
def load_numpy_compatible(data_dict):
    """numpy 버전 호환 형식에서 배열 복원"""
    if isinstance(data_dict, dict) and 'data' in data_dict and 'shape' in data_dict:
        return np.frombuffer(data_dict['data'], dtype=data_dict['dtype']).reshape(data_dict['shape'])
    return data_dict  # 이미 numpy 배열인 경우 (구버전 pickle)

# 추천 기능 테스트
def recommend_for_user(model, user_id, k=10, exclude_known=True):
    """
    사용자에게 제품 추천
    
    Args:
        model: 로드된 Pickle 모델
        user_id: 사용자 ID
        k: 추천 개수
        exclude_known: 이미 상호작용한 제품 제외 여부
    
    Returns:
        list: 추천 제품 ID 리스트
    """
    components = model['components']
    
    # 사용자 인덱스 확인
    user_idx = components['user_id_to_idx'].get(user_id)
    
    if user_idx is None:
        # 콜드 스타트: 글로벌 인기 제품 반환
        return components['global_popular_products'][:k]
    
    # 임베딩 복원 (numpy 호환 형식 처리)
    user_embeddings = load_numpy_compatible(components['user_embeddings'])
    product_embeddings = load_numpy_compatible(components['product_embeddings'])
    
    # 사용자 임베딩
    user_vec = user_embeddings[user_idx]
    
    # 모든 제품과의 유사도 계산 (코사인 유사도 = 내적, 이미 L2 정규화됨)
    scores = np.dot(product_embeddings, user_vec)
    
    # Top-K 인덱스
    top_k_indices = np.argsort(scores)[-k:][::-1]
    
    # 제품 ID로 변환
    recommendations = [
        components['idx_to_product_id'][idx] 
        for idx in top_k_indices
    ]
    
    return recommendations


# 테스트 실행
print("추천 기능 테스트:")

# 존재하는 사용자로 테스트
test_user_ids = list(loaded_model['components']['user_id_to_idx'].keys())[:3]

for user_id in test_user_ids:
    recommendations = recommend_for_user(loaded_model, user_id, k=5)
    print(f"\n사용자 {user_id}에 대한 추천:")
    print(f"  {recommendations}")

# 존재하지 않는 사용자 (콜드 스타트)
cold_user_id = 999999
cold_recommendations = recommend_for_user(loaded_model, cold_user_id, k=5)
print(f"\n콜드 스타트 사용자 {cold_user_id}에 대한 추천 (글로벌 인기):")
print(f"  {cold_recommendations}")

추천 기능 테스트:

사용자 1에 대한 추천:
  [1665, 376, 168, 1401, 426]

사용자 2에 대한 추천:
  [1184, 1759, 649, 189, 1206]

사용자 3에 대한 추천:
  [205, 380, 661, 834, 1391]

콜드 스타트 사용자 999999에 대한 추천 (글로벌 인기):
  [1124, 789, 1759, 324, 1526]


In [ ]:
# 추천 속도 벤치마크
import time

print("추천 속도 벤치마크:")

n_tests = 100
test_users = list(loaded_model['components']['user_id_to_idx'].keys())[:n_tests]

start_time = time.time()
for user_id in test_users:
    _ = recommend_for_user(loaded_model, user_id, k=10)
elapsed = time.time() - start_time

avg_time_ms = (elapsed / n_tests) * 1000

print(f"  테스트 횟수: {n_tests}")
print(f"  총 소요 시간: {elapsed:.3f}초")
print(f"  평균 추천 시간: {avg_time_ms:.2f}ms")

# 성능 기준 확인 (50ms 이하)
if avg_time_ms < 50:
    print(f"  ✅ 성능 기준 통과 (< 50ms)")
else:
    print(f"  ⚠️ 성능 개선 필요 (> 50ms)")

추천 속도 벤치마크:
  테스트 횟수: 100
  총 소요 시간: 0.007초
  평균 추천 시간: 0.07ms
  ✅ 성능 기준 통과 (< 50ms)


## 7. 모델 정보 요약

In [ ]:
# 최종 요약
print("=" * 60)
print("SelF Personalized Model Pickle 생성 완료")
print("=" * 60)

print(f"\n📁 출력 파일: {OUTPUT_PATH}")
print(f"📊 파일 크기: {file_size_mb:.2f} MB")

print(f"\n📈 모델 정보:")
print(f"   버전: {loaded_model['version']}")
print(f"   생성일: {loaded_model['created_at']}")
print(f"   사용자 수: {loaded_model['metadata']['n_users']:,}")
print(f"   제품 수: {loaded_model['metadata']['n_products']:,}")
print(f"   임베딩 차원: {loaded_model['metadata']['embedding_dim']}")

print(f"\n⚙️ 하이퍼파라미터:")
for key, value in loaded_model['hyperparameters'].items():
    print(f"   {key}: {value}")

if EVAL_LOADED:
    print(f"\n📊 평가 메트릭:")
    for key, value in loaded_model['metadata']['evaluation_metrics'].items():
        if isinstance(value, float):
            print(f"   {key}: {value:.4f}")
        else:
            print(f"   {key}: {value}")

print(f"\n⚡ 성능:")
print(f"   평균 추천 시간: {avg_time_ms:.2f}ms")

print("\n" + "=" * 60)
print("✅ Pickle 생성 및 검증 완료")
print("=" * 60)

SelF Personalized Model Pickle 생성 완료

📁 출력 파일: ..\pred\models\self_personalized_v1.pkl
📊 파일 크기: 3.50 MB

📈 모델 정보:
   버전: 1.0.0
   생성일: 2025-12-15 02:00:50.360387
   사용자 수: 5,000
   제품 수: 1,995
   임베딩 차원: 128

⚙️ 하이퍼파라미터:
   n_components: 128
   weight_view: 1.0
   weight_cart: 3.0
   weight_order: 5.0
   log_transform: True

📊 평가 메트릭:
   same_category_ratio: 0.1014
   hit_at_10: 0.0088
   ndcg_at_10: 0.0022
   all_passed: False

⚡ 성능:
   평균 추천 시간: 0.07ms

✅ Pickle 생성 및 검증 완료


## 8. 검증 체크리스트

### 필수 확인 항목
- [ ] SVD 결과 파일 로드 성공
- [ ] 데이터 일관성 검증 통과
- [ ] Pickle 파일 저장 완료
- [ ] 저장된 Pickle 로드 검증 통과
- [ ] 추천 기능 테스트 성공
- [ ] 추천 속도 < 50ms

### 다음 단계
1. **pred 서비스 코드 업데이트**: `pred/ml/self_personalized.py`에서 이 Pickle 로드
2. **API 엔드포인트 테스트**: `/api/recommendations/personalized/` 호출 확인
3. **Phase 3 진행**: Price Anomaly 모델 개발 (30번 노트북부터)

### 사용 방법
```python
import pickle

# 모델 로드
with open('pred/models/self_personalized_v1.pkl', 'rb') as f:
    model = pickle.load(f)

# 추천 생성
user_id = 123
user_idx = model['components']['user_id_to_idx'].get(user_id)

if user_idx is not None:
    user_vec = model['components']['user_embeddings'][user_idx]
    scores = np.dot(model['components']['product_embeddings'], user_vec)
    top_k = np.argsort(scores)[-10:][::-1]
    recommendations = [model['components']['idx_to_product_id'][i] for i in top_k]
else:
    # 콜드 스타트 폴백
    recommendations = model['components']['global_popular_products'][:10]
```